This notebook evaluates the sentence classifier on the hold-out test set.

In [1]:
from dap_job_quality.pipeline.find_job_quality import JobQuality, split_into_chunks

from datasets import Dataset
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import time

job_quality = JobQuality()
job_quality.load()

2024-09-03 16:42:01,158 - datasets - INFO - PyTorch version 2.1.2 available.


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-09-03 16:42:05,628 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-09-03 16:42:05,799 - root - INFO - Loading models and variables
2024-09-03 16:42:05,949 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2024-09-03 16:42:06,202 - root - INFO - Downloading the model...
2024-09-03 16:42:54,471 - root - INFO - Loading the model and tokenizer...
2024-09-03 16:42:54,820 - aiobotocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


2024-09-03 16:42:55,181 - root - INFO - Calculating embeddings for 131 target phrases ...


Batches: 100%|██████████| 5/5 [00:00<00:00,  7.33it/s]


In [15]:
sets = ['train', 'val', 'test']

datasets = {}

for set in sets:
    print(f'Loading {set} set...')
    datasets[set] = pd.read_parquet(f's3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/{set}_df_20240725.parquet')
    

# test_df = pd.read_parquet('s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/test_df_20240725.parquet')

Loading train set...
Loading val set...
Loading test set...


In [16]:
output_dict = {}

for set, df in datasets.items():
    
    df = df[df['sentence'].notna()] # there were 3 NAs in the training set! Not sure how this happened
    
    dataset = Dataset.from_pandas(df)

    # Make sure there are no super super long sentences
    df["chunks"] = df["sentence"].apply(
                lambda x: split_into_chunks(x) if len(x.split()) >= 25 else [x]
            )
    df = df.explode("chunks").reset_index(drop=True)
    df = df.drop(columns=["sentence"])
    df = df.rename(columns={"chunks": "sentence"})

    dataset = Dataset.from_pandas(df)

    start_time = time.time()
    predictions = job_quality.job_quality_classifier(
                dataset["sentence"], batch_size=job_quality.batch_size
            )

    elapsed_time = time.time() - start_time
    print(f"Time taken: {elapsed_time:.2f} seconds")

    labels = []
    pred_scores = []
    for pred in predictions:
        labels.append(pred["label"])
        pred_scores.append(pred["score"])

    df["job_quality_label"] = labels
    df["job_quality_prob"] = pred_scores
    
    # Rename 'LABEL_0' to 0 and 'LABEL_1' to 1 in the 'job_quality_label' column
    df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})
    
    output_dict[set] = df

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_54502/163966962.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["chunks"] = df["sentence"].apply(


Time taken: 66.36 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_54502/163966962.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


Time taken: 12.93 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_54502/163966962.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


Time taken: 14.47 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_54502/163966962.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


In [21]:
classification_reports = {}

for set, df in output_dict.items():
    conf_matrix = confusion_matrix(df['label'], df['job_quality_label'])
    print(f'{set} set confusion matrix:')
    print(conf_matrix)
    classification_reports[set] = pd.DataFrame(classification_report(df['label'], df['job_quality_label'], output_dict=True))

train set confusion matrix:
[[748  85]
 [285 828]]
val set confusion matrix:
[[155  12]
 [ 42 149]]
test set confusion matrix:
[[150  20]
 [ 55 219]]


In [22]:
classification_reports['train']

,0,1,accuracy,macro avg,weighted avg
precision,0.724105,0.906900,0.809866,0.815502,0.828653
recall,0.897959,0.743935,0.809866,0.820947,0.809866
f1-score,0.801715,0.817374,0.809866,0.809545,0.810671
support,833.000000,1113.000000,0.809866,1946.000000,1946.000000


In [23]:
classification_reports['val']

,0,1,accuracy,macro avg,weighted avg
precision,0.786802,0.925466,0.849162,0.856134,0.860782
recall,0.928144,0.780105,0.849162,0.854124,0.849162
f1-score,0.851648,0.846591,0.849162,0.849120,0.848950
support,167.000000,191.000000,0.849162,358.000000,358.000000


In [24]:
classification_reports['test']

,0,1,accuracy,macro avg,weighted avg
precision,0.731707,0.916318,0.831081,0.824013,0.845634
recall,0.882353,0.799270,0.831081,0.840812,0.831081
f1-score,0.800000,0.853801,0.831081,0.826901,0.833202
support,170.000000,274.000000,0.831081,444.000000,444.000000
